# Duplicate gematria in Genesis 1

Find full written words with the same numerical value in the first chapter of Genesis, using **Mispar Hechrechi** and **Mispar Gadol**. Load the published gematria features from `tonyjurg/gematria_TF`, alongside their matching BHSA source, through Text-Fabric.

## Table of content (ToC)<a class="anchor" id="TOC"></a>
* <a href="#bullet1">1 - Introduction and comparison policy</a>
* <a href="#bullet2">2 - Setup</a>
    * <a href="#bullet2x1">2.1 - Dependencies</a>
    * <a href="#bullet2x2">2.2 - Download and verify the TF datasets</a>
    * <a href="#bullet2x3">2.3 - Load the features</a>
* <a href="#bullet3">3 - Select Genesis 1</a>
* <a href="#bullet4">4 - Find duplicate word values</a>
* <a href="#bullet5">5 - Show the matching words and verses</a>
* <a href="#bullet6">6 - Checks, attribution, and notebook version</a>


# 1 - Introduction and comparison policy <a class="anchor" id="bullet1"></a>
##### [Back to ToC](#TOC)

A duplicate means that **at least two word occurrences in Genesis 1 share a value under the same method**. A match can occur within one verse or across verses. The table lists every verse containing each duplicated value. It does not compare whole-verse totals, or treat equal values under different methods as a match.

We use the dataset's **full written form**, **ketiv**, and **identity transformation**. Attached prefixes and suffix letters already in the text are included. BHSA divides some written words into multiple slots, so `gem_full_start_ketiv` selects each written form once. Maqaf ends a form under this dataset's documented grouping convention.

Repeated spellings count as duplicates too. Set `ONLY_DIFFERENT_FORMS = True` below to restrict the table to equal values shared by different spellings. Hechrechi gives final letters their ordinary values; Gadol gives them 500–900. Numerical equality alone does not establish an interpretive relationship.


# 2 - Setup <a class="anchor" id="bullet2"></a>
##### [Back to ToC](#TOC)

## 2.1 - Dependencies <a class="anchor" id="bullet2x1"></a>
##### [Back to ToC](#TOC)

Use Python 3.11 or newer. In the repository's virtual environment, install and open Jupyter:

```console
python -m pip install -r requirements-notebooks.txt
python -m jupyter lab genesis1_duplicate_gematria.ipynb
```

Select that environment's Python kernel and use **Run → Run All Cells**. The notebook is also portable: its code does not import project-local modules and can run from another folder with Text-Fabric and IPython installed. No gematria is recalculated; the example reads the published `.tf` features.


In [1]:
from collections import Counter, defaultdict
from html import escape
from pathlib import Path
import hashlib
import json

from IPython.display import HTML, display
from tf.advanced.repo import checkoutRepo
from tf.fabric import Fabric

DATA_VERSION = "2021"
# Pin the published dataset snapshot; newer notebook commits need not change this.
GEMATRIA_COMMIT = "f3bf2e009ba9ac0a2991ab1272e167a28fbdc426"
ONLY_DIFFERENT_FORMS = False
METHODS = {
    "Hechrechi": "gem_hechrechi_full_ketiv_ident",
    "Gadol": "gem_gadol_full_ketiv_ident",
}


## 2.2 - Download and verify the TF datasets <a class="anchor" id="bullet2x2"></a>
##### [Back to ToC](#TOC)

Text-Fabric downloads both datasets into its standard `~/text-fabric-data` tree and reuses that copy. The gematria manifest supplies the exact matching BHSA commit and source checksums. SHA-256 checks verify the required BHSA files and the gematria features used here.

Public repository downloads can be anonymous, subject to GitHub's API rate limit. Authenticated downloads use the optional `GHPERS` environment variable; browser login alone does not set it. A repository that is private additionally requires authorized access. Keep credentials outside notebooks and their outputs. See [Increase the GitHub rate limit](https://nbviewer.org/github/CenterBLC/N1904/blob/main/docs/tutorial/Increase_GitHub_rate_limit.ipynb).


In [2]:
def checkout_tf(org, repo, commit, folder="/tf"):
    _, _, _, base, relative = checkoutRepo(
        "github", org=org, repo=repo, folder=folder, version=DATA_VERSION,
        checkout=commit, allowExpress=False, silent="deep",
    )
    if not base or not relative:
        raise RuntimeError(f"Text-Fabric could not obtain {org}/{repo}; check download access.")
    directory = (Path(base).expanduser() / relative / DATA_VERSION).resolve()
    if not directory.is_dir():
        raise RuntimeError(f"Downloaded TF directory was not found: {directory}")
    return directory


def file_digest(path, canonical_lf=False):
    data = path.read_bytes()
    if canonical_lf:
        data = data.replace(b"\r\n", b"\n")
    return hashlib.sha256(data).hexdigest()


addon = checkout_tf("tonyjurg", "gematria_TF", GEMATRIA_COMMIT)
provenance = checkout_tf("tonyjurg", "gematria_TF", GEMATRIA_COMMIT, folder="/provenance")
manifest = json.loads((provenance / "manifest.json").read_text(encoding="utf-8"))
source_info = manifest["source"]
if source_info["version"] != DATA_VERSION or manifest["limit"] is not None:
    raise ValueError("Expected the complete BHSA 2021 gematria dataset.")
source = checkout_tf("etcbc", "bhsa", source_info["commit"])

for filename, expected in source_info["sha256"].items():
    if file_digest(source / filename, canonical_lf=True) != expected:
        raise ValueError(f"BHSA checksum mismatch: {filename}")

feature_names = list(METHODS.values()) + ["gem_text_full_ketiv", "gem_full_start_ketiv"]
for feature in feature_names:
    filename = f"{feature}.tf"
    if file_digest(addon / filename, canonical_lf=True) != manifest["features"][filename]:
        raise ValueError(f"Gematria checksum mismatch: {filename}")
print(f"Verified gematria dataset {manifest['generatorVersion']} and BHSA {DATA_VERSION}.")


unexpected error from builtins: 'RateLimitOverview' object has no attribute 'core'


unexpected error from builtins: 'RateLimitOverview' object has no attribute 'core'


unexpected error from builtins: 'RateLimitOverview' object has no attribute 'core'


Verified gematria dataset 0.2.0 and BHSA 2021.


## 2.3 - Load the features <a class="anchor" id="bullet2x3"></a>
##### [Back to ToC](#TOC)

Load the two numerical features, the readable full-form text and group-start feature, and BHSA's passage features. Text-Fabric's section and locality APIs will select the chapter, verses, and their word nodes.


In [3]:
TF = Fabric(locations=[str(source), str(addon)], silent="deep")
api = TF.load(feature_names + ["book", "chapter", "verse"], silent="deep")
if not api:
    raise RuntimeError("Text-Fabric could not load BHSA and the gematria features.")
F, L, T = api.F, api.L, api.T


# 3 - Select Genesis 1 <a class="anchor" id="bullet3"></a>
##### [Back to ToC](#TOC)

Ask Text-Fabric for the chapter by its passage reference. Retain only each full form's first slot, so attached prefixes do not create artificial duplicates. Each occurrence retains its BHSA word node and verse reference for inspection.


In [4]:
chapter_node = T.nodeFromSection(("Genesis", 1))
if chapter_node is None:
    raise ValueError("Genesis 1 was not found in this BHSA source.")
verse_nodes = L.d(chapter_node, otype="verse")
occurrences = []
for verse_node in verse_nodes:
    reference = T.sectionFromNode(verse_node)
    for word_node in L.d(verse_node, otype="word"):
        if F.gem_full_start_ketiv.v(word_node) != word_node:
            continue
        text = F.gem_text_full_ketiv.v(word_node)
        values = {method: getattr(F, feature).v(word_node)
                  for method, feature in METHODS.items()}
        if not text or any(value is None for value in values.values()):
            raise ValueError(f"Missing full-form data at BHSA word node {word_node}.")
        occurrences.append({"node": word_node, "verse": reference[2],
                            "text": text, "values": values})

assert len(verse_nodes) == 31
assert {item["verse"] for item in occurrences} == set(range(1, 32))
assert len({item["node"] for item in occurrences}) == len(occurrences)
print(f"Selected {len(verse_nodes)} verses and {len(occurrences)} full written word occurrences.")


Selected 31 verses and 434 full written word occurrences.


# 4 - Find duplicate word values <a class="anchor" id="bullet4"></a>
##### [Back to ToC](#TOC)

Group by **method and value**. Keep groups containing at least two occurrences in this chapter. The optional different-spellings filter is applied after grouping. Every matching verse is listed once, even when it contains several matching occurrences.


In [5]:
duplicate_groups = []
for method in METHODS:
    by_value = defaultdict(list)
    for item in occurrences:
        by_value[item["values"][method]].append(item)
    for value, members in sorted(by_value.items()):
        forms = Counter(item["text"] for item in members)
        if len(members) < 2 or (ONLY_DIFFERENT_FORMS and len(forms) < 2):
            continue
        duplicate_groups.append({
            "method": method, "value": value, "forms": forms,
            "count": len(members), "verses": sorted({item["verse"] for item in members}),
            "nodes": [item["node"] for item in members],
        })
print(f"Found {len(duplicate_groups)} duplicated values across the two methods.")


Found 155 duplicated values across the two methods.


# 5 - Show the matching words and verses <a class="anchor" id="bullet5"></a>
##### [Back to ToC](#TOC)

Each row is one duplicated value under one method. Words include their occurrence counts; the final column lists **all matching verses in Genesis 1**. Separate Hechrechi and Gadol rows are intentional. An empty result is displayed explicitly rather than interpreted as a failed load.


In [6]:
rows = []
for group in duplicate_groups:
    words = "; ".join(
        f'<span lang="he" dir="rtl">{escape(form)}</span> × {count}'
        for form, count in sorted(group["forms"].items())
    )
    verses = ", ".join(f"1:{verse}" for verse in group["verses"])
    rows.append(
        f"<tr><td>{escape(group['method'])}</td><td>{group['value']}</td>"
        f"<td>{words}</td><td>{group['count']}</td><td>{verses}</td></tr>"
    )
if rows:
    result_html = (
        '<div style="overflow-x:auto"><table>'
        '<caption>Duplicate full-form ketiv gematria — Genesis 1</caption>'
        '<thead><tr><th>Method</th><th>Gematria</th><th>Words × occurrences</th>'
        '<th>Total occurrences</th><th>Genesis verses</th></tr></thead>'
        '<tbody>' + "".join(rows) + '</tbody></table></div>'
    )
else:
    result_html = "<p>No duplicate word values were found with the selected policy.</p>"
display(HTML(result_html))


Method,Gematria,Words × occurrences,Total occurrences,Genesis verses
Hechrechi,8,בו × 4,4,"1:11, 1:12, 1:29, 1:30"
Hechrechi,13,אחד × 2,2,"1:5, 1:9"
Hechrechi,17,טוב × 7,7,"1:4, 1:10, 1:12, 1:18, 1:21, 1:25, 1:31"
Hechrechi,23,חיה × 4,4,"1:20, 1:24, 1:28, 1:30"
Hechrechi,25,יהי × 3,3,"1:3, 1:6, 1:14"
Hechrechi,27,והיו × 2,2,"1:14, 1:15"
Hechrechi,30,יהיה × 1; כי × 6,7,"1:4, 1:10, 1:12, 1:18, 1:21, 1:25, 1:29"
Hechrechi,31,אל × 1; ויהי × 20,21,"1:3, 1:5, 1:6, 1:7, 1:8, 1:9, 1:11, 1:13, 1:15, 1:19, 1:23, 1:24, 1:30, 1:31"
Hechrechi,45,אדם × 1; מאד × 1,2,"1:26, 1:31"
Hechrechi,50,האדם × 1; כל × 8,9,"1:21, 1:25, 1:27, 1:29, 1:30, 1:31"


# 6 - Checks, attribution, and notebook version <a class="anchor" id="bullet6"></a>
##### [Back to ToC](#TOC)

These checks confirm the chapter selection, counting policy, and known standard values. The first verse's total is used only as a check; the table above compares **word values**.


In [7]:
assert occurrences[0]["text"] == "בראשית"
assert occurrences[0]["values"]["Hechrechi"] == 913
assert sum(item["values"]["Hechrechi"] for item in occurrences if item["verse"] == 1) == 2701
for group in duplicate_groups:
    assert group["count"] == len(group["nodes"]) == sum(group["forms"].values())
    assert group["count"] >= 2
    assert all(1 <= verse <= 31 for verse in group["verses"])
    matching = [item for item in occurrences if item["values"][group["method"]] == group["value"]]
    assert group["nodes"] == [item["node"] for item in matching]
    assert group["verses"] == sorted({item["verse"] for item in matching})
    if ONLY_DIFFERENT_FORMS:
        assert len(group["forms"]) >= 2
print("Chapter, known values, duplicate groups, and complete verse listings verified.")


Chapter, known values, duplicate groups, and complete verse listings verified.


BHSA: ETCBC, [DOI 10.17026/dans-z6y-skyh](https://doi.org/10.17026/dans-z6y-skyh), CC BY-NC 4.0. Gematria data: [gematria_TF](https://github.com/tonyjurg/gematria_TF), CC BY-NC 4.0. Read the project's [representation policy](https://github.com/tonyjurg/gematria_TF/blob/main/docs/representations.md) and [methods](https://github.com/tonyjurg/gematria_TF/blob/main/docs/methods.md).

The clickable ToC, numbered sections, subsections, and Back to ToC links follow Tony Jurg's [create_tfidf_features.ipynb](https://github.com/tonyjurg/Create_TF-IDF_Text-Fabric_features/blob/main/create_tfidf_features.ipynb). The analysis code is newly written. Notebook prose/workflow: CC BY 4.0; Python code: MIT; displayed BHSA-derived words: CC BY-NC 4.0.

| Notebook | Author | Version | Date |
|---|---|---|---|
| Genesis 1 duplicate gematria | Tony Jurg | 1.0 | 5 October 2026 |

[Back to ToC](#TOC)
